# Support Vector Machines with Palmer Penguins



## Libraries and Data



In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from plotnine.data import penguins 
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold



# Load penguin data
df = penguins.copy().reset_index(drop=True).dropna()

features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]


X = df[features]
y = df["species"]



## Train Test Split


In [ ]:
# Preserve approximately the same species proportions in each set
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=909,
)

## Standardization of Data and SVM Model

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
    )),
])

## Fitting Model

In [ ]:
svm_model.fit(X_train, y_train)

### Predicting Test Data

In [ ]:
y_pred = svm_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Cross-Validation Setup

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=2434,
)

### Setting Up Parameter Grid

In [ ]:
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": [0.1, 1, 10, 100],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale", 0.001, 0.01, 0.1, 1],
    },
]

### Grid Search

In [ ]:

search = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=2,
    refit=True
)

search.fit(X_train, y_train)

### Best Model

In [ ]:




print("Best parameters:", search.best_params_)
print("Best cross-validation accuracy:", search.best_score_)

# Best pipeline, refitted using all training observations
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Printing Confusion Matrix

In [ ]:


species = best_model.named_steps["svm"].classes_

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=species),
    index=pd.Index(species, name="Actual"),
    columns=pd.Index(species, name="Predicted"),
)

print(confusion)